In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import optuna
from sklearn.metrics import average_precision_score
from collections import defaultdict
import itertools

############################## CONFIGURATION PARAMETERS - MODIFY THESE!

# Optuna Optimization Settings
OPTUNA_CONFIG = {
    'n_trials': 500,                    # Reduced due to increased complexity
    'timeout': None,                   
    'random_state': 42,                 
    'n_jobs': 4,                        # Reduced due to memory constraints
}

# SIFT Parameter Search Space
SIFT_PARAMS = {
    'n_features': [0, 3000],           
    'n_octaves': [2, 16],              
    'edge_threshold': [2.0, 30.0],     
    'contrast_threshold': [0.005, 0.5], 
    'sigma': [0.5, 5],               
}

# Matcher Configuration
MATCHER_CONFIG = {
    'types': ['bf', 'flann'],          
    'lowe_ratio': [0.55, 0.9],         
    'flann_trees': [1, 10],           
    'flann_checks': [10, 100],        
}

# Feature Processing Settings
FEATURE_CONFIG = {
    'enable_precise_upscale': True,    
    'cross_check': False,              
}

# Variant Configuration
VARIANT_CONFIG = {
    'variant_names': ['original', 'inverted', '128x128', '256x256', '400x400', 
                     '1024x1024', '2048x2048', 'gray', 'histeq', 'sharpen', 'blur'],
    'enable_weight_optimization': True,  # Whether to optimize variant weights
    'min_variant_weight': 0.1,          # Minimum weight for any variant
    'max_variant_weight': 2.0,          # Maximum weight for any variant
}

# WRRF Configuration
WRRF_CONFIG = {
    'k': 60,                           # Parameter k in WRRF formula
    'enable_k_optimization': True,     # Whether to optimize k parameter
    'min_k': 10,                       # Minimum k value
    'max_k': 100,                      # Maximum k value
}

# Evaluation Settings
EVAL_CONFIG = {
    'interpolation_points': 11,        
    'min_positive_samples': 1,         
}

############################## IMAGE LOADING AND VARIANT GENERATION

def safe_imread(p):
    img = cv2.imread(p)
    if img is None: 
        raise IOError(f"Could not load image: {p}")
    return img

def make_variants_from_img(img, variant_names):
    """Generate multiple variants of an image for robust feature extraction"""
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    variants = {}
    
    for variant in variant_names:
        if variant == 'original':
            variants[variant] = img.copy()
        elif variant == 'inverted':
            variants[variant] = cv2.bitwise_not(img)
        elif variant == '128x128':
            variants[variant] = cv2.resize(img, (128, 128), interpolation=cv2.INTER_LINEAR)
        elif variant == '256x256':
            variants[variant] = cv2.resize(img, (256, 256), interpolation=cv2.INTER_LINEAR)
        elif variant == '400x400':
            variants[variant] = cv2.resize(img, (400, 400), interpolation=cv2.INTER_LINEAR)
        elif variant == '1024x1024':
            variants[variant] = cv2.resize(img, (1024, 1024), interpolation=cv2.INTER_LINEAR)
        elif variant == '2048x2048':
            variants[variant] = cv2.resize(img, (2048, 2048), interpolation=cv2.INTER_LINEAR)
        elif variant == 'gray':
            variants[variant] = g
        elif variant == 'histeq':
            variants[variant] = cv2.equalizeHist(g)
        elif variant == 'sharpen':
            kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
            variants[variant] = cv2.filter2D(img, -1, kernel)
        elif variant == 'blur':
            variants[variant] = cv2.GaussianBlur(img, (5, 5), 0)
    
    return variants

# Load and process images
print("Loading images and generating variants...")

# Read query images
query_images = []
logo_names = []
query_dir = sorted(os.listdir('queries/query/'))
for img_name in query_dir:
    img = cv2.imread(os.path.join('queries/query/', img_name))
    img = cv2.resize(img, dsize=(400, 400))
    query_images.append(img)
    logo_names.append(os.path.splitext(img_name)[0][:-5].capitalize())

# Read db images  
db_images = []
db_dir = sorted(os.listdir('database/'), key=lambda x: int(x.split('.')[0]))
for img_name in db_dir:
    img = cv2.imread(os.path.join('database/', img_name))
    img = cv2.resize(img, dsize=(400, 400))
    db_images.append(img)

print(f"Loaded {len(query_images)} query images and {len(db_images)} database images")

# Generate variants for all images
query_variants = {v: [] for v in VARIANT_CONFIG['variant_names']}
db_variants = {v: [] for v in VARIANT_CONFIG['variant_names']}

for img in query_images:
    variants = make_variants_from_img(img, VARIANT_CONFIG['variant_names'])
    for v in VARIANT_CONFIG['variant_names']:
        query_variants[v].append(variants[v])

for img in db_images:
    variants = make_variants_from_img(img, VARIANT_CONFIG['variant_names'])
    for v in VARIANT_CONFIG['variant_names']:
        db_variants[v].append(variants[v])

print("Image variants generated successfully!")

# Ground-truth arrays per logo
coca_cola_GT = np.zeros(len(db_images))
marlboro_GT = np.zeros(len(db_images))
starbucks_GT = np.zeros(len(db_images))
heineken_GT = np.zeros(len(db_images))
coca_cola_GT[18:25] = 1
marlboro_GT[55:61] = 1
marlboro_GT[100:] = 1
starbucks_GT[70:100] = 1
heineken_GT[25:55] = 1

GT_matrix = np.vstack([coca_cola_GT, heineken_GT, marlboro_GT, starbucks_GT])
print(f"GT matrix shape: {GT_matrix.shape}")

############################## WEIGHTED RECIPROCAL RANK FUSION FUNCTIONS

def weighted_reciprocal_rank_fusion(rank_lists, weights, k=60):
    """
    Perform Weighted Reciprocal Rank Fusion
    
    Parameters:
    - rank_lists: list of lists, each containing database indices ranked by similarity
    - weights: list of weights for each rank list
    - k: parameter in RRF formula
    
    Returns:
    - fused_scores: combined scores for each database image
    """
    n_db = len(db_images)
    fused_scores = np.zeros(n_db)
    
    for i, (ranks, weight) in enumerate(zip(rank_lists, weights)):
        for rank_pos, db_idx in enumerate(ranks):
            # RRF formula: weight / (k + rank_position)
            fused_scores[db_idx] += weight / (k + rank_pos + 1)
    
    return fused_scores

def get_ranked_list_from_scores(scores):
    """Convert similarity scores to ranked list of indices"""
    return np.argsort(scores)[::-1]  # Descending order

############################## UPDATED OPTUNA OBJECTIVE FUNCTION

def create_objective_function(config):
    def objective(trial):
        # Suggest SIFT parameters (common for all variants)
        n_features = trial.suggest_int('n_features', 
                                     config['SIFT_PARAMS']['n_features'][0],
                                     config['SIFT_PARAMS']['n_features'][1])
        
        n_octaves = trial.suggest_int('n_octaves',
                                    config['SIFT_PARAMS']['n_octaves'][0],
                                    config['SIFT_PARAMS']['n_octaves'][1])
        
        edge_threshold = trial.suggest_float('edge_threshold',
                                           config['SIFT_PARAMS']['edge_threshold'][0],
                                           config['SIFT_PARAMS']['edge_threshold'][1])
        
        contrast_threshold = trial.suggest_float('contrast_threshold',
                                               config['SIFT_PARAMS']['contrast_threshold'][0],
                                               config['SIFT_PARAMS']['contrast_threshold'][1])
        
        sigma = trial.suggest_float('sigma',
                                  config['SIFT_PARAMS']['sigma'][0],
                                  config['SIFT_PARAMS']['sigma'][1])
        
        # Suggest matcher parameters
        matcher_type = trial.suggest_categorical('matcher_type', 
                                               config['MATCHER_CONFIG']['types'])
        
        lowe_ratio = trial.suggest_float('lowe_ratio',
                                       config['MATCHER_CONFIG']['lowe_ratio'][0],
                                       config['MATCHER_CONFIG']['lowe_ratio'][1])
        
        # Suggest WRRF parameters
        if config['WRRF_CONFIG']['enable_k_optimization']:
            k = trial.suggest_int('wrrf_k',
                                config['WRRF_CONFIG']['min_k'],
                                config['WRRF_CONFIG']['max_k'])
        else:
            k = config['WRRF_CONFIG']['k']
        
        # Suggest variant weights
        variant_weights = {}
        if config['VARIANT_CONFIG']['enable_weight_optimization']:
            for variant in config['VARIANT_CONFIG']['variant_names']:
                weight = trial.suggest_float(
                    f'weight_{variant}',
                    config['VARIANT_CONFIG']['min_variant_weight'],
                    config['VARIANT_CONFIG']['max_variant_weight']
                )
                variant_weights[variant] = weight
        else:
            # Equal weights if not optimizing
            for variant in config['VARIANT_CONFIG']['variant_names']:
                variant_weights[variant] = 1.0
        
        # Create detector
        detector = cv2.SIFT_create(
            nfeatures=n_features,
            nOctaveLayers=n_octaves,
            edgeThreshold=edge_threshold,
            contrastThreshold=contrast_threshold,
            sigma=sigma
        )
        
        # Compute descriptors for all variants
        query_des_by_variant = {v: [] for v in config['VARIANT_CONFIG']['variant_names']}
        db_des_by_variant = {v: [] for v in config['VARIANT_CONFIG']['variant_names']}
        
        for variant in config['VARIANT_CONFIG']['variant_names']:
            for img in query_variants[variant]:
                _, des = detector.detectAndCompute(img, None)
                query_des_by_variant[variant].append(des)
            
            for img in db_variants[variant]:
                _, des = detector.detectAndCompute(img, None)
                db_des_by_variant[variant].append(des)
        
        # Create matcher
        if matcher_type == 'bf':
            matcher = cv2.BFMatcher_create(cv2.NORM_L2, 
                                         crossCheck=config['FEATURE_CONFIG']['cross_check'])
        else:
            flann_trees = trial.suggest_int('flann_trees',
                                          config['MATCHER_CONFIG']['flann_trees'][0],
                                          config['MATCHER_CONFIG']['flann_trees'][1])
            flann_checks = trial.suggest_int('flann_checks',
                                           config['MATCHER_CONFIG']['flann_checks'][0],
                                           config['MATCHER_CONFIG']['flann_checks'][1])
            
            index_params = dict(algorithm=1, trees=flann_trees)
            search_params = dict(checks=flann_checks)
            matcher = cv2.FlannBasedMatcher(index_params, search_params)
        
        # For each query, perform matching across all variants and fuse results
        all_query_scores = []
        
        for q_idx in range(len(query_images)):
            variant_rank_lists = []
            variant_weights_list = []
            
            # Get ranked lists for each variant
            for variant in config['VARIANT_CONFIG']['variant_names']:
                if (query_des_by_variant[variant][q_idx] is None or 
                    len(query_des_by_variant[variant][q_idx]) == 0):
                    continue
                
                # Compute similarity scores for this variant
                variant_scores = np.zeros(len(db_images))
                
                for db_idx in range(len(db_images)):
                    if (db_des_by_variant[variant][db_idx] is None or 
                        len(db_des_by_variant[variant][db_idx]) == 0):
                        variant_scores[db_idx] = 0
                        continue
                    
                    try:
                        matches = matcher.knnMatch(
                            query_des_by_variant[variant][q_idx], 
                            db_des_by_variant[variant][db_idx], 
                            k=2
                        )
                        
                        n_matches = 0
                        for match_pair in matches:
                            if len(match_pair) == 2:
                                m, n = match_pair
                                if m.distance < lowe_ratio * n.distance:
                                    n_matches += 1
                        
                        variant_scores[db_idx] = n_matches
                    except Exception as e:
                        variant_scores[db_idx] = 0
                
                # Convert scores to ranked list
                ranked_list = get_ranked_list_from_scores(variant_scores)
                variant_rank_lists.append(ranked_list)
                variant_weights_list.append(variant_weights[variant])
            
            # Skip if no valid variants found matches
            if not variant_rank_lists:
                all_query_scores.append(np.zeros(len(db_images)))
                continue
            
            # Fuse rankings using WRRF
            fused_scores = weighted_reciprocal_rank_fusion(
                variant_rank_lists, 
                variant_weights_list, 
                k=k
            )
            
            all_query_scores.append(fused_scores)
        
        # Calculate mAP
        AP_all = np.zeros(len(query_images))
        valid_queries = 0
        
        for i in range(len(query_images)):
            if len(all_query_scores[i]) > 0:
                scores = all_query_scores[i]
                gt = GT_matrix[i]
                
                if np.sum(gt) >= config['EVAL_CONFIG']['min_positive_samples']:
                    ap = average_precision_score(gt, scores)
                    AP_all[i] = ap
                    valid_queries += 1
        
        mAP = np.sum(AP_all) / valid_queries if valid_queries > 0 else 0.0
        
        # Store additional information
        trial.set_user_attr('valid_queries', valid_queries)
        trial.set_user_attr('used_variants', len(variant_weights))
        trial.set_user_attr('wrrf_k', k)
        
        return mAP
    
    return objective

############################## RUN OPTIMIZATION

# Combine all configurations
CONFIG = {
    'OPTUNA_CONFIG': OPTUNA_CONFIG,
    'SIFT_PARAMS': SIFT_PARAMS,
    'MATCHER_CONFIG': MATCHER_CONFIG,
    'FEATURE_CONFIG': FEATURE_CONFIG,
    'VARIANT_CONFIG': VARIANT_CONFIG,
    'WRRF_CONFIG': WRRF_CONFIG,
    'EVAL_CONFIG': EVAL_CONFIG,
}

print("Starting multi-variant SIFT optimization with WRRF...")

# Create study
sampler = optuna.samplers.TPESampler(seed=OPTUNA_CONFIG['random_state'])
study = optuna.create_study(direction='maximize', sampler=sampler)

# Run optimization
study.optimize(
    create_objective_function(CONFIG),
    n_trials=OPTUNA_CONFIG['n_trials'],
    timeout=OPTUNA_CONFIG['timeout'],
    n_jobs=OPTUNA_CONFIG['n_jobs']
)

print("Optimization completed!")
print(f"Number of completed trials: {len(study.trials)}")

############################## DISPLAY RESULTS

print("\n" + "="*60)
print("MULTI-VARIANT SIFT + WRRF OPTIMIZATION RESULTS")
print("="*60)

best_trial = study.best_trial
print(f"Best mAP: {best_trial.value:.4f}")
print(f"Best trial number: {best_trial.number}")

print("\nBest hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

print(f"\nAdditional info:")
print(f"  Valid queries: {best_trial.user_attrs.get('valid_queries', 'N/A')}")
print(f"  Used variants: {best_trial.user_attrs.get('used_variants', 'N/A')}")
print(f"  WRRF k: {best_trial.user_attrs.get('wrrf_k', 'N/A')}")

# Display variant weights
print(f"\nVariant weights:")
for key, value in best_trial.params.items():
    if key.startswith('weight_'):
        variant_name = key.replace('weight_', '')
        print(f"  {variant_name}: {value:.3f}")

# Plot optimization history
try:
    fig = optuna.visualization.plot_optimization_history(study)
    fig.show()
except Exception as e:
    print(f"Could not plot optimization history: {e}")

# Plot parameter importance
try:
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
except Exception as e:
    print(f"Could not plot parameter importance: {e}")

############################## FINAL EVALUATION FUNCTION

def evaluate_final_model(best_params, config):
    """Evaluate the final model with best parameters"""
    print("\n" + "="*50)
    print("FINAL MODEL EVALUATION")
    print("="*50)
    
    # Extract parameters
    n_features = best_params['n_features']
    n_octaves = best_params['n_octaves']
    edge_threshold = best_params['edge_threshold']
    contrast_threshold = best_params['contrast_threshold']
    sigma = best_params['sigma']
    matcher_type = best_params['matcher_type']
    lowe_ratio = best_params['lowe_ratio']
    k = best_params.get('wrrf_k', config['WRRF_CONFIG']['k'])
    
    # Extract variant weights
    variant_weights = {}
    for variant in config['VARIANT_CONFIG']['variant_names']:
        weight_key = f'weight_{variant}'
        if weight_key in best_params:
            variant_weights[variant] = best_params[weight_key]
        else:
            variant_weights[variant] = 1.0
    
    # Create detector and matcher
    detector = cv2.SIFT_create(
        nfeatures=n_features,
        nOctaveLayers=n_octaves,
        edgeThreshold=edge_threshold,
        contrastThreshold=contrast_threshold,
        sigma=sigma
    )
    
    if matcher_type == 'bf':
        matcher = cv2.BFMatcher_create(cv2.NORM_L2, crossCheck=config['FEATURE_CONFIG']['cross_check'])
    else:
        flann_trees = best_params.get('flann_trees', 5)
        flann_checks = best_params.get('flann_checks', 50)
        index_params = dict(algorithm=1, trees=flann_trees)
        search_params = dict(checks=flann_checks)
        matcher = cv2.FlannBasedMatcher(index_params, search_params)
    
    # Compute descriptors for all variants
    query_des_by_variant = {v: [] for v in config['VARIANT_CONFIG']['variant_names']}
    db_des_by_variant = {v: [] for v in config['VARIANT_CONFIG']['variant_names']}
    
    print("Computing descriptors for all variants...")
    for variant in config['VARIANT_CONFIG']['variant_names']:
        for img in query_variants[variant]:
            _, des = detector.detectAndCompute(img, None)
            query_des_by_variant[variant].append(des)
        
        for img in db_variants[variant]:
            _, des = detector.detectAndCompute(img, None)
            db_des_by_variant[variant].append(des)
    
    # Perform matching and fusion
    print("Performing matching and WRRF fusion...")
    all_query_scores = []
    individual_variant_aps = {v: [] for v in config['VARIANT_CONFIG']['variant_names']}
    
    for q_idx in range(len(query_images)):
        variant_rank_lists = []
        variant_weights_list = []
        variant_scores_dict = {}
        
        # Get ranked lists for each variant
        for variant in config['VARIANT_CONFIG']['variant_names']:
            if (query_des_by_variant[variant][q_idx] is None or 
                len(query_des_by_variant[variant][q_idx]) == 0):
                variant_scores_dict[variant] = np.zeros(len(db_images))
                continue
            
            # Compute similarity scores for this variant
            variant_scores = np.zeros(len(db_images))
            
            for db_idx in range(len(db_images)):
                if (db_des_by_variant[variant][db_idx] is None or 
                    len(db_des_by_variant[variant][db_idx]) == 0):
                    variant_scores[db_idx] = 0
                    continue
                
                try:
                    matches = matcher.knnMatch(
                        query_des_by_variant[variant][q_idx], 
                        db_des_by_variant[variant][db_idx], 
                        k=2
                    )
                    
                    n_matches = 0
                    for match_pair in matches:
                        if len(match_pair) == 2:
                            m, n = match_pair
                            if m.distance < lowe_ratio * n.distance:
                                n_matches += 1
                    
                    variant_scores[db_idx] = n_matches
                except Exception as e:
                    variant_scores[db_idx] = 0
            
            variant_scores_dict[variant] = variant_scores
            ranked_list = get_ranked_list_from_scores(variant_scores)
            variant_rank_lists.append(ranked_list)
            variant_weights_list.append(variant_weights[variant])
            
            # Calculate AP for individual variant
            gt = GT_matrix[q_idx]
            if np.sum(gt) >= config['EVAL_CONFIG']['min_positive_samples']:
                ap = average_precision_score(gt, variant_scores)
                individual_variant_aps[variant].append(ap)
        
        # Fuse rankings using WRRF
        if variant_rank_lists:
            fused_scores = weighted_reciprocal_rank_fusion(
                variant_rank_lists, 
                variant_weights_list, 
                k=k
            )
            all_query_scores.append(fused_scores)
        else:
            all_query_scores.append(np.zeros(len(db_images)))
    
    # Calculate final mAP
    AP_all = np.zeros(len(query_images))
    valid_queries = 0
    
    for i in range(len(query_images)):
        if len(all_query_scores[i]) > 0:
            scores = all_query_scores[i]
            gt = GT_matrix[i]
            
            if np.sum(gt) >= config['EVAL_CONFIG']['min_positive_samples']:
                ap = average_precision_score(gt, scores)
                AP_all[i] = ap
                valid_queries += 1
    
    final_mAP = np.sum(AP_all) / valid_queries if valid_queries > 0 else 0.0
    
    print(f"\nFinal Results:")
    print(f"Multi-variant WRRF mAP: {final_mAP:.4f}")
    
    # Print individual variant performance
    print(f"\nIndividual variant performance:")
    for variant in config['VARIANT_CONFIG']['variant_names']:
        if individual_variant_aps[variant]:
            variant_mAP = np.mean(individual_variant_aps[variant])
            weight = variant_weights[variant]
            print(f"  {variant:12s}: mAP = {variant_mAP:.4f}, weight = {weight:.3f}")
    
    return final_mAP, all_query_scores

# Run final evaluation
final_mAP, final_scores = evaluate_final_model(best_trial.params, CONFIG)

print("\n" + "="*60)
print("OPTIMIZATION COMPLETED SUCCESSFULLY!")
print("="*60)

/home/walterjtv/.pyenv/versions/3.12.11/envs/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading images and generating variants...
Loaded 4 query images and 110 database images


[I 2025-11-09 21:16:59,099] A new study created in memory with name: no-name-c63c1713-f65c-4859-938c-955da807f623


Image variants generated successfully!
GT matrix shape: (4, 110)
Starting multi-variant SIFT optimization with WRRF...


[I 2025-11-09 21:22:27,662] Trial 1 finished with value: 0.3583448550478614 and parameters: {'n_features': 776, 'n_octaves': 11, 'edge_threshold': 13.584929251941187, 'contrast_threshold': 0.3750692937013851, 'sigma': 1.5011396726494928, 'matcher_type': 'bf', 'lowe_ratio': 0.7182388773929366, 'wrrf_k': 94, 'weight_original': 1.7557081217862585, 'weight_inverted': 0.49260928740540244, 'weight_128x128': 1.552169182193933, 'weight_256x256': 1.2382132962668302, 'weight_400x400': 1.0460613214639585, 'weight_1024x1024': 0.6802769058314816, 'weight_2048x2048': 1.6238031704751343, 'weight_gray': 0.16755712899710887, 'weight_histeq': 0.9842521920386773, 'weight_sharpen': 0.584324243299331, 'weight_blur': 1.4217260833982468}. Best is trial 1 with value: 0.3583448550478614.
[I 2025-11-09 21:23:06,969] Trial 3 finished with value: 0.392334261175451 and parameters: {'n_features': 381, 'n_octaves': 10, 'edge_threshold': 14.669371864438371, 'contrast_threshold': 0.35232593150949215, 'sigma': 2.492219